<div style="background:linear-gradient(90deg,#0b3d91,#1f6fb2);color:#fff;padding:22px 26px;border-radius:10px">
<h1 style="margin:0;color:#fff">Python for Data Science — Hands-On Workshop</h1>
<h3 style="margin:6px 0 0;color:#dfe9f7">Session 2 of 4 · Numerical Computing with NumPy</h3>
<p style="margin:10px 0 0">📅 <b>Date:</b> 16/11/2026 &nbsp;|&nbsp; 👤 <b>Instructor:</b> Mr. Mahesh Divakaran &nbsp;|&nbsp; 🏷️ <b>Batch:</b> 30</p>
</div>

### 🎯 What you will learn today

| # | Topic | You will be able to… |
|---|-------|----------------------|
| 1 | Why NumPy | see why arrays are faster than Python lists |
| 2 | Creating arrays | make arrays with `array`, `zeros`, `ones`, `arange`, `linspace` |
| 3 | Array attributes | read `shape`, `ndim`, `dtype`, `size` |
| 4 | Indexing & masking | pick values, rows, columns and filter with conditions |
| 5 | Vectorised maths & ufuncs | do maths on a whole array in one line |
| 6 | Axis aggregations | compute per-patient and per-visit averages (`axis=0` vs `axis=1`) |
| 7 | Missing values | handle `np.nan` and `np.inf` safely |
| 8 | Broadcasting | combine arrays of different shapes |
| 9 | Reshaping & stacking | change shape and join arrays |
| 10 | Random numbers | simulate reproducible data with `np.random.default_rng(42)` |

### 🧭 How to use this notebook
- Run each cell with **Shift + Enter**, from top to bottom.
- Read the text cell first, then run the code, then read **"What to observe"**.
- Change the numbers and run again — that is the best way to learn!

> ✏️ Theme: **lab biomarker readings** (blood glucose, cholesterol) for patients across clinic visits. In Session 1 we stored readings in lists — now they become NumPy arrays.

## 0. Importing NumPy

NumPy is a library for working with **numbers in bulk**. Everyone imports it with the short name `np`.

In [1]:
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.5.3


> 👀 **What to observe:** Any version 2.x is fine for this session.

## 1. Why NumPy? List vs Array

With a Python **list**, maths needs a loop. With a NumPy **array**, you write the maths once and it applies to every value.

| | Python list | NumPy array |
|---|---|---|
| Add 10 to every value | loop or comprehension | `arr + 10` |
| Speed on 1 million values | slow | very fast (written in C) |
| Memory | each number is a separate object | numbers packed together |
| Data types | can mix types | one type for all values |

In [2]:
glucose_list = [95, 110, 132, 88, 145]          # mg/dL, Session 1 style

# List: need a loop to convert mg/dL to mmol/L
glucose_mmol_list = [round(g / 18, 1) for g in glucose_list]
print("List result :", glucose_mmol_list)

# Array: one line, no loop
glucose = np.array(glucose_list)
glucose_mmol = np.round(glucose / 18, 1)
print("Array result:", glucose_mmol)

List result : [5.3, 6.1, 7.3, 4.9, 8.1]
Array result: [5.3 6.1 7.3 4.9 8.1]


> 👀 **What to observe:** Same answer, but the array version has no loop. That is called **vectorisation**.

### 1.1 Speed Test

Let's time both ways on **1 million** readings using `time.perf_counter()` (a stopwatch).

In [3]:
import time

big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)

start = time.perf_counter()
result_list = [x * 2 for x in big_list]
list_time = time.perf_counter() - start

start = time.perf_counter()
result_array = big_array * 2
array_time = time.perf_counter() - start

print(f"List time : {list_time * 1000:.1f} ms")
print(f"Array time: {array_time * 1000:.1f} ms")
print(f"NumPy is about {list_time / array_time:.0f} times faster")

List time : 34.2 ms
Array time: 4.3 ms
NumPy is about 8 times faster


> 👀 **What to observe:** Times differ on each computer, but NumPy is usually **10–100 times faster**.

### 1.2 The Jupyter way: `%%timeit`

In Jupyter / Colab, `%%timeit` at the top of a cell runs it many times and reports the average.
(This is a Jupyter-only command, so the cell below shows no saved output — run it yourself.)

In [4]:
%%timeit
big_array * 2

> 👀 **What to observe:** You will see something like `500 µs ± 20 µs per loop`. Try the same with `[x * 2 for x in big_list]` in a new cell.

## 2. Creating Arrays

| Function | Makes | Example |
|----------|-------|---------|
| `np.array(list)` | array from a list | `np.array([95, 110])` |
| `np.zeros(n)` | all zeros | empty score sheet |
| `np.ones(n)` | all ones | default weights |
| `np.full(n, value)` | all the same value | baseline dose |
| `np.arange(start, stop, step)` | evenly stepped numbers (stop not included) | visit days |
| `np.linspace(start, stop, num)` | `num` evenly spaced numbers (stop included) | dose levels |

In [5]:
cholesterol = np.array([180, 210, 195, 240, 170])
print("From a list :", cholesterol)

print("Zeros       :", np.zeros(5))
print("Ones        :", np.ones(3))
print("Full of 10  :", np.full(4, 10))

visit_days = np.arange(0, 29, 7)          # day 0, 7, 14, 21, 28
print("Visit days  :", visit_days)

doses = np.linspace(0, 100, 5)            # 5 dose levels from 0 to 100 mg
print("Dose levels :", doses)

From a list : [180 210 195 240 170]
Zeros       : [0. 0. 0. 0. 0.]
Ones        : [1. 1. 1.]
Full of 10  : [10 10 10 10]
Visit days  : [ 0  7 14 21 28]
Dose levels : [  0.  25.  50.  75. 100.]


> 👀 **What to observe:** `arange(0, 29, 7)` stops **before** 29. `linspace(0, 100, 5)` **includes** 100 and gives exactly 5 values.

### 2.1 2-D Arrays (Tables)

A 2-D array is like a table: **rows × columns**. Here each **row is a patient** and each **column is a visit**.

In [6]:
# Blood glucose (mg/dL): 4 patients (rows) x 3 visits (columns)
glucose_table = np.array([
    [ 95, 102,  98],     # patient 1
    [140, 132, 125],     # patient 2
    [110, 115, 108],     # patient 3
    [160, 150, 138],     # patient 4
])
print(glucose_table)

[[ 95 102  98]
 [140 132 125]
 [110 115 108]
 [160 150 138]]


> 👀 **What to observe:** A list of lists becomes a neat grid with 4 rows and 3 columns.

### 2.2 Array Attributes

| Attribute | Tells you | For `glucose_table` |
|-----------|-----------|---------------------|
| `.shape` | (rows, columns) | `(4, 3)` |
| `.ndim` | number of dimensions | `2` |
| `.size` | total number of values | `12` |
| `.dtype` | data type of the values | `int64` |

In [7]:
print("shape:", glucose_table.shape)
print("ndim :", glucose_table.ndim)
print("size :", glucose_table.size)
print("dtype:", glucose_table.dtype)

# Mixing int and float -> everything becomes float
mixed = np.array([1, 2.5, 3])
print("\nmixed:", mixed, "dtype:", mixed.dtype)

shape: (4, 3)
ndim : 2
size : 12
dtype: int64

mixed: [1.  2.5 3. ] dtype: float64


> 👀 **What to observe:** All values in an array share **one** dtype. Mixing integers and decimals turns everything into `float64`. (On Windows the integer type may show as `int32` — that's fine.)

## 3. Indexing and Slicing

### 3.1 1-D Arrays
Works just like lists: positions start at **0**, and `-1` is the last value.

In [8]:
cholesterol = np.array([180, 210, 195, 240, 170])

print("First        :", cholesterol[0])
print("Last         :", cholesterol[-1])
print("First three  :", cholesterol[0:3])
print("Every second :", cholesterol[::2])

First        : 180
Last         : 170
First three  : [180 210 195]
Every second : [180 195 170]


> 👀 **What to observe:** Same rules as Session 1 lists — `[0:3]` gives positions 0, 1, 2.

### 3.2 2-D Arrays: `array[row, column]`

| Code | Meaning |
|------|---------|
| `t[1, 2]` | row 1, column 2 (one value) |
| `t[0]` or `t[0, :]` | the whole first row |
| `t[:, 1]` | the whole second column |
| `t[1:3, 0:2]` | a block of rows 1–2 and columns 0–1 |

In [9]:
print("Patient 2, visit 3 :", glucose_table[1, 2])
print("Patient 1 all visits:", glucose_table[0])
print("Visit 2 all patients:", glucose_table[:, 1])
print("Patients 2-3, visits 1-2:")
print(glucose_table[1:3, 0:2])

Patient 2, visit 3 : 125
Patient 1 all visits: [ 95 102  98]
Visit 2 all patients: [102 132 115 150]
Patients 2-3, visits 1-2:
[[140 132]
 [110 115]]


> 👀 **What to observe:** `:` means **all**. `[:, 1]` = all rows, column 1 = everyone's second visit.

### 3.3 Fancy Indexing — Pick Several Positions

Pass a **list of positions** to grab exactly the values you want.

In [10]:
patient_ids = np.array(["P001", "P002", "P003", "P004", "P005"])
cholesterol = np.array([180, 210, 195, 240, 170])

pick = [1, 3]                    # positions we want
print(patient_ids[pick], cholesterol[pick])

# Rows 0 and 3 of the glucose table
print(glucose_table[[0, 3]])

['P002' 'P004'] [210 240]
[[ 95 102  98]
 [160 150 138]]


> 👀 **What to observe:** You get P002 and P004 — the positions in the list, in that order.

### 3.4 Boolean Masking — Filter with a Condition

A condition on an array gives an array of `True` / `False` (a **mask**). Using the mask keeps only the `True` positions.

Combine conditions with `&` (and), `|` (or) and `~` (not). **Put brackets around each condition.**

In [11]:
cholesterol = np.array([180, 210, 195, 240, 170])
patient_ids = np.array(["P001", "P002", "P003", "P004", "P005"])

high_mask = cholesterol >= 200
print("Mask           :", high_mask)
print("High values    :", cholesterol[high_mask])
print("High patients  :", patient_ids[high_mask])
print("How many high? :", high_mask.sum())        # True counts as 1

# Two conditions: between 180 and 210
borderline = cholesterol[(cholesterol >= 180) & (cholesterol <= 210)]
print("180 to 210     :", borderline)

Mask           : [False  True False  True False]
High values    : [210 240]
High patients  : ['P002' 'P004']
How many high? : 2
180 to 210     : [180 210 195]


> 👀 **What to observe:** `True` counts as 1, so `.sum()` counts how many values passed. Forgetting the brackets in `(a) & (b)` gives an error.

In [12]:
# Masks work on 2-D arrays too: all glucose readings above 126 mg/dL
print("Readings > 126:", glucose_table[glucose_table > 126])

Readings > 126: [140 132 160 150 138]


> 👀 **What to observe:** The result is a flat 1-D array of the matching values.

## 4. Vectorised Maths and Universal Functions (ufuncs)

Arithmetic (`+ - * / **`) works on **every value at once**.
**ufuncs** are NumPy functions that also work on every value: `np.sqrt`, `np.round`, `np.log`, `np.abs`, `np.exp`, `np.sin` …

In [13]:
weight_kg = np.array([68.0, 82.5, 55.0, 90.2])
height_m  = np.array([1.62, 1.75, 1.58, 1.80])

bmi = weight_kg / height_m ** 2              # value-by-value maths
print("BMI           :", np.round(bmi, 1))

print("Weight in lb  :", np.round(weight_kg * 2.2046, 1))
print("Square root   :", np.round(np.sqrt(weight_kg), 2))
print("Log of weight :", np.round(np.log(weight_kg), 2))

change = np.array([-5, 3, -2, 8])
print("Absolute change:", np.abs(change))

BMI           : [25.9 26.9 22.  27.8]
Weight in lb  : [149.9 181.9 121.3 198.9]
Square root   : [8.25 9.08 7.42 9.5 ]
Log of weight : [4.22 4.41 4.01 4.5 ]
Absolute change: [5 3 2 8]


> 👀 **What to observe:** No loops anywhere. Two arrays of the **same length** are combined position by position.

In [14]:
# Trigonometry example: body temperature rises and falls over 24 hours
hours = np.arange(0, 24, 6)                          # 0, 6, 12, 18
temp = 36.8 + 0.4 * np.sin(2 * np.pi * hours / 24)  # simple daily rhythm
print("Hours:", hours)
print("Temp :", np.round(temp, 2))

Hours: [ 0  6 12 18]
Temp : [36.8 37.2 36.8 36.4]


> 👀 **What to observe:** `np.sin` and `np.pi` work on the whole `hours` array at once.

## 5. Statistics and the `axis` Argument

| Function | Gives |
|----------|-------|
| `np.mean` / `np.median` | average / middle value |
| `np.std` | spread (standard deviation) |
| `np.min` / `np.max` | lowest / highest |
| `np.percentile(a, q)` | value below which `q`% of data falls |

For a 2-D table:
- **`axis=0`** → go **down the rows** → one answer **per column** (per visit)
- **`axis=1`** → go **across the columns** → one answer **per row** (per patient)

In [15]:
cholesterol = np.array([180, 210, 195, 240, 170])

print("Mean     :", np.mean(cholesterol))
print("Median   :", np.median(cholesterol))
print("Std dev  :", round(np.std(cholesterol), 1))
print("Min / Max:", np.min(cholesterol), "/", np.max(cholesterol))
print("90th percentile:", np.percentile(cholesterol, 90))

Mean     : 199.0
Median   : 195.0
Std dev  : 24.6
Min / Max: 170 / 240
90th percentile: 228.0


In [16]:
print(glucose_table)
print("\nMean per visit   (axis=0):", np.round(glucose_table.mean(axis=0), 1))
print("Mean per patient (axis=1):", np.round(glucose_table.mean(axis=1), 1))
print("Overall mean     (no axis):", round(glucose_table.mean(), 1))

[[ 95 102  98]
 [140 132 125]
 [110 115 108]
 [160 150 138]]

Mean per visit   (axis=0): [126.2 124.8 117.2]
Mean per patient (axis=1): [ 98.3 132.3 111.  149.3]
Overall mean     (no axis): 122.8


> 👀 **What to observe:** `axis=0` gives **3** answers (one per visit). `axis=1` gives **4** answers (one per patient). No axis gives a single number for the whole table.

## 6. Missing and Invalid Values: `np.nan` and `np.inf`

- `np.nan` = **Not a Number** → a missing reading.
- `np.inf` = **infinity** → often from dividing by zero.

⚠️ Any maths with `nan` gives `nan`. Use the **nan-safe** functions: `np.nanmean`, `np.nanmax`, `np.nansum`…
Check for missing values with `np.isnan()`. Replace values with `np.where(condition, if_true, if_false)`.

In [17]:
hba1c = np.array([5.6, 6.2, np.nan, 7.1, np.nan, 5.9])    # two missing results

print("Normal mean   :", np.mean(hba1c))        # nan - one missing spoils it
print("nan-safe mean :", round(np.nanmean(hba1c), 2))
print("Missing mask  :", np.isnan(hba1c))
print("How many missing:", np.isnan(hba1c).sum())

# Fill missing values with the average of the others
filled = np.where(np.isnan(hba1c), np.nanmean(hba1c), hba1c)
print("Filled        :", np.round(filled, 2))

Normal mean   : nan
nan-safe mean : 6.2
Missing mask  : [False False  True False  True False]
How many missing: 2
Filled        : [5.6 6.2 6.2 7.1 6.2 5.9]


In [18]:
# np.where for labels
hba1c = np.array([5.6, 6.2, 7.1, 5.9, 6.8])
status = np.where(hba1c >= 6.5, "Diabetic range", "Below 6.5")
print(status)

# Dividing by zero gives inf (NumPy warns but does not crash)
visits_done = np.array([4, 0, 2])
doses_taken = np.array([8, 3, 2])
with np.errstate(divide="ignore"):
    doses_per_visit = doses_taken / visits_done
print("Doses per visit:", doses_per_visit)
print("Is finite?     :", np.isfinite(doses_per_visit))

['Below 6.5' 'Below 6.5' 'Diabetic range' 'Below 6.5' 'Diabetic range']
Doses per visit: [ 2. inf  1.]
Is finite?     : [ True False  True]


> 👀 **What to observe:** `np.mean` returns `nan`, `np.nanmean` ignores the gaps. `np.where` works like an `if/else` for a whole array. `np.isfinite` flags `inf` and `nan`.

## 7. Broadcasting — Arrays of Different Shapes

NumPy can stretch a smaller array to match a bigger one, **without copying data**.

**Rule (simple version):** compare shapes from the **right**. Two sizes fit if they are **equal** or one of them is **1**.

| Big table | Small array | Works? | Why |
|-----------|-------------|--------|-----|
| `(4, 3)` | single number | ✅ | a number fits everything |
| `(4, 3)` | `(3,)` | ✅ | last sizes match (3 = 3) → applied to every row |
| `(4, 3)` | `(4, 1)` | ✅ | 1 stretches → applied to every column |
| `(4, 3)` | `(4,)` | ❌ | 3 ≠ 4 |

In [19]:
print(glucose_table, glucose_table.shape)

# 1) Table and a single number
print("\nmg/dL -> mmol/L:")
print(np.round(glucose_table / 18, 1))

# 2) Table (4,3) and one value per VISIT (3,): subtract each visit's lab calibration offset
calibration = np.array([2, -1, 0])
print("\nAfter calibration:")
print(glucose_table - calibration)

[[ 95 102  98]
 [140 132 125]
 [110 115 108]
 [160 150 138]] (4, 3)

mg/dL -> mmol/L:
[[5.3 5.7 5.4]
 [7.8 7.3 6.9]
 [6.1 6.4 6. ]
 [8.9 8.3 7.7]]

After calibration:
[[ 93 103  98]
 [138 133 125]
 [108 116 108]
 [158 151 138]]


In [20]:
# 3) Table (4,3) and one value per PATIENT (4,1): change from each patient's own baseline
baseline = glucose_table[:, 0].reshape(4, 1)     # first visit as a column
print("Baseline shape:", baseline.shape)
print("Change from baseline:")
print(glucose_table - baseline)

# 4) A shape that does not fit
try:
    glucose_table - np.array([1, 2, 3, 4])
except ValueError as error:
    print("\nError:", error)

Baseline shape: (4, 1)
Change from baseline:
[[  0   7   3]
 [  0  -8 -15]
 [  0   5  -2]
 [  0 -10 -22]]

Error: operands could not be broadcast together with shapes (4,3) (4,) 


> 👀 **What to observe:** The first column of 'change from baseline' is all zeros — each patient compared with themselves. A `(4,)` array cannot broadcast to `(4, 3)`, so NumPy raises a clear error.

## 8. Reshaping and Stacking

| Function | What it does |
|----------|--------------|
| `a.reshape(rows, cols)` | new shape, same values (sizes must multiply to `a.size`) |
| `a.reshape(-1, cols)` | `-1` = "work out this number for me" |
| `a.flatten()` | make 1-D, always a **copy** |
| `a.ravel()` | make 1-D, a **view** when possible (faster, shares data) |
| `np.vstack([a, b])` | stack **vertically** (add rows) |
| `np.hstack([a, b])` | stack **horizontally** (add columns) |

In [21]:
readings = np.arange(1, 13)            # 12 readings in a row
print("Original:", readings, readings.shape)

table = readings.reshape(4, 3)         # 4 patients x 3 visits
print("\nreshape(4, 3):")
print(table)

print("\nreshape(-1, 2) shape:", readings.reshape(-1, 2).shape)
print("flatten:", table.flatten())
print("ravel  :", table.ravel())

Original: [ 1  2  3  4  5  6  7  8  9 10 11 12] (12,)

reshape(4, 3):
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]

reshape(-1, 2) shape: (6, 2)
flatten: [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


In [22]:
# flatten() is a copy, ravel() can share data
table = np.arange(1, 7).reshape(2, 3)
flat_copy = table.flatten()
flat_view = table.ravel()
flat_copy[0] = 100
flat_view[1] = 200
print("Original after changes:")
print(table)

Original after changes:
[[  1 200   3]
 [  4   5   6]]


> 👀 **What to observe:** Changing the **ravel** result changed the original (200 appears); changing the **flatten** copy did not (no 100).

In [23]:
site_a = np.array([[95, 102], [140, 132]])      # 2 patients x 2 visits
site_b = np.array([[110, 115], [160, 150]])

print("vstack (more patients):")
print(np.vstack([site_a, site_b]))

visit_3 = np.array([[98], [125]])               # one new column for site A
print("\nhstack (add a visit):")
print(np.hstack([site_a, visit_3]))

vstack (more patients):
[[ 95 102]
 [140 132]
 [110 115]
 [160 150]]

hstack (add a visit):
[[ 95 102  98]
 [140 132 125]]


> 👀 **What to observe:** `vstack` gives 4 rows; `hstack` gives 3 columns. Column counts must match for `vstack`, row counts for `hstack`.

## 9. Random Numbers with `np.random.default_rng()`

Simulated data is perfect for practice and testing. Always create a **generator** with a **seed**, so results are the same every time.

| Method | Gives |
|--------|-------|
| `rng.normal(mean, sd, size)` | bell-curve values (e.g. cholesterol) |
| `rng.uniform(low, high, size)` | evenly spread values |
| `rng.integers(low, high, size)` | whole numbers (high not included) |
| `rng.choice(options, size)` | random picks from a list |

> Old code uses `np.random.seed()` and `np.random.rand()`. The modern way is `rng = np.random.default_rng(42)`.

In [24]:
rng = np.random.default_rng(42)

cholesterol_sim = rng.normal(loc=200, scale=25, size=8)      # mean 200, SD 25
print("Normal  :", np.round(cholesterol_sim, 1))

adherence = rng.uniform(60, 100, size=5)                      # % of doses taken
print("Uniform :", np.round(adherence, 1))

ages = rng.integers(18, 76, size=6)                           # 18 to 75
print("Integers:", ages)

arms = rng.choice(["Drug A", "Placebo"], size=6)
print("Choice  :", arms)

Normal  : [207.6 174.  218.8 223.5 151.2 167.4 203.2 192.1]
Uniform : [65.1 78.  74.8 97.1 85.8]
Integers: [41 65 49 43 44 31]
Choice  : ['Drug A' 'Placebo' 'Placebo' 'Drug A' 'Placebo' 'Placebo']


In [25]:
# Reproducibility: same seed -> same numbers
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)
print(rng1.integers(1, 100, size=5))
print(rng2.integers(1, 100, size=5))

rng3 = np.random.default_rng(7)      # different seed -> different numbers
print(rng3.integers(1, 100, size=5))

[ 9 77 65 44 43]
[ 9 77 65 44 43]
[94 62 68 89 58]


> 👀 **What to observe:** The first two lines are **identical** because the seed is the same. Everyone in the room gets the same 'random' data.

### 9.1 Simulating a Biomarker Study

100 patients × 4 visits of LDL cholesterol (mg/dL), with values slowly falling over time on treatment.

In [26]:
rng = np.random.default_rng(42)

n_patients, n_visits = 100, 4
baseline_ldl = rng.normal(160, 20, size=(n_patients, 1))         # one column
weekly_drop = np.array([0, 8, 15, 20])                             # mg/dL, per visit
noise = rng.normal(0, 5, size=(n_patients, n_visits))

ldl = baseline_ldl - weekly_drop + noise                            # broadcasting!
print("Shape:", ldl.shape)
print("First 3 patients:")
print(np.round(ldl[:3], 1))
print("\nMean LDL per visit:", np.round(ldl.mean(axis=0), 1))
print("Patients below 130 at last visit:", (ldl[:, -1] < 130).sum())

Shape: (100, 4)
First 3 patients:
[[164.2 164.6 149.3 149.8]
 [134.5 130.2 119.5 117.5]
 [179.2 158.4 162.2 156.2]]

Mean LDL per visit: [159.  150.6 144.  139.3]
Patients below 130 at last visit: 29


> 👀 **What to observe:** Mean LDL goes down visit by visit (about 160 → 140). Broadcasting combined a `(100, 1)` baseline, a `(4,)` drop and a `(100, 4)` noise table in one line.

---
## 🧪 Practice Exercises

### Exercise 1 — Beginner: Heart rate array
1. Create an array `heart_rate = np.array([72, 88, 95, 60, 110, 78])`.
2. Print its `shape`, `mean`, `min` and `max`.
3. Using a **mask**, print the readings above 90 and how many there are.

In [27]:
# Exercise 1 - write your code here

### Exercise 2 — Intermediate: Blood pressure table
```python
bp = np.array([
    [150, 145, 138],
    [128, 126, 122],
    [160, 152, 141],
    [118, 120, 117],
])   # 4 patients x 3 visits (systolic, mmHg)
```
1. Print the mean per **visit** and the mean per **patient** (round to 1 decimal).
2. Using **broadcasting**, compute each patient's change from their **first visit**.
3. Using `np.where`, label every value `"High"` (≥ 140) or `"OK"`.

In [28]:
# Exercise 2 - write your code here

### Exercise 3 — Challenge: Simulate and clean
1. With `rng = np.random.default_rng(42)`, simulate **50 patients × 3 visits** of fasting glucose: `rng.normal(110, 15, size=(50, 3))`.
2. Set 5 random positions to `np.nan` (hint: `rng.integers(0, 50, 5)` for rows and `rng.integers(0, 3, 5)` for columns).
3. Count the missing values, then compute the nan-safe mean per visit.
4. Fill each missing value with **its visit's** mean (hint: `np.where` + broadcasting).
5. Stack a new 4th visit (`rng.normal(105, 15, size=(50, 1))`) with `np.hstack` and print the final shape.

In [29]:
# Exercise 3 - write your code here

---
## ✅ Solutions
The solution cells are **collapsed** — open them after you try.

In [30]:
#@title Solution 1 — Heart rate array
heart_rate = np.array([72, 88, 95, 60, 110, 78])

print("Shape:", heart_rate.shape)
print("Mean :", round(heart_rate.mean(), 1))
print("Min  :", heart_rate.min(), "| Max:", heart_rate.max())

fast = heart_rate[heart_rate > 90]
print("Above 90:", fast, "| count:", len(fast))

Shape: (6,)
Mean : 83.8
Min  : 60 | Max: 110
Above 90: [ 95 110] | count: 2


In [31]:
#@title Solution 2 — Blood pressure table
bp = np.array([
    [150, 145, 138],
    [128, 126, 122],
    [160, 152, 141],
    [118, 120, 117],
])

print("Mean per visit  :", np.round(bp.mean(axis=0), 1))
print("Mean per patient:", np.round(bp.mean(axis=1), 1))

change = bp - bp[:, [0]]          # bp[:, [0]] keeps a (4, 1) column shape
print("Change from first visit:")
print(change)

print(np.where(bp >= 140, "High", "OK"))

Mean per visit  : [139.  135.8 129.5]
Mean per patient: [144.3 125.3 151.  118.3]
Change from first visit:
[[  0  -5 -12]
 [  0  -2  -6]
 [  0  -8 -19]
 [  0   2  -1]]
[['High' 'High' 'OK']
 ['OK' 'OK' 'OK']
 ['High' 'High' 'High']
 ['OK' 'OK' 'OK']]


In [32]:
#@title Solution 3 — Simulate and clean
rng = np.random.default_rng(42)
glucose = rng.normal(110, 15, size=(50, 3))

rows = rng.integers(0, 50, 5)
cols = rng.integers(0, 3, 5)
glucose[rows, cols] = np.nan

print("Missing values:", np.isnan(glucose).sum())   # may be < 5 if a position repeats

visit_means = np.nanmean(glucose, axis=0)            # shape (3,)
print("nan-safe mean per visit:", np.round(visit_means, 1))

glucose_filled = np.where(np.isnan(glucose), visit_means, glucose)   # (50,3) with (3,)
print("Missing after fill:", np.isnan(glucose_filled).sum())

visit_4 = rng.normal(105, 15, size=(50, 1))
glucose_all = np.hstack([glucose_filled, visit_4])
print("Final shape:", glucose_all.shape)

Missing values: 4
nan-safe mean per visit: [110.2 107.8 109.4]
Missing after fill: 0
Final shape: (50, 4)


---
### 📌 Session 2 Summary

| Topic | Remember |
|-------|----------|
| Arrays | one data type, fast, maths without loops |
| Create | `array`, `zeros`, `ones`, `arange` (stop excluded), `linspace` (stop included) |
| Attributes | `shape`, `ndim`, `size`, `dtype` |
| Select | `a[row, col]`, `a[:, 1]`, `a[[0, 3]]`, `a[a > 100]` with `&`, `|`, `~` |
| Statistics | `axis=0` per column, `axis=1` per row |
| Missing | `np.nan` spoils maths → use `np.nanmean`, `np.isnan`, `np.where` |
| Broadcasting | compare shapes from the right: equal or 1 |
| Random | `rng = np.random.default_rng(42)` |

**Next — Session 3: Pandas**, where arrays get column names and become DataFrames. 🚀